# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import f1_score, confusion_matrix, classification_report, precision_recall_curve, auc
from imblearn.over_sampling import SMOTE


# Load Cleaned Data

In [2]:
fraud_df = pd.read_csv('../data/Fraud_Data.csv')         
credit_df = pd.read_csv('../data/creditcard.csv')       


# Data Preparation – E-Commerce (fraud_df)

In [3]:
# Drop unnecessary columns
X_fraud = fraud_df.drop(columns=['class', 'signup_time', 'purchase_time', 'ip_address', 'user_id', 'device_id'])
y_fraud = fraud_df['class']

# One-hot encode categorical features
X_fraud = pd.get_dummies(X_fraud, drop_first=True)

# Train-Test split
Xf_train, Xf_test, yf_train, yf_test = train_test_split(X_fraud, y_fraud, stratify=y_fraud, test_size=0.2, random_state=42)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
Xf_train_resampled, yf_train_resampled = smote.fit_resample(Xf_train, yf_train)


# Data Preparation – Credit Card (credit_df)

In [4]:
X_credit = credit_df.drop(columns=['Class'])
y_credit = credit_df['Class']

# Normalize Amount column
scaler = StandardScaler()
X_credit['Amount'] = scaler.fit_transform(X_credit[['Amount']])

# Train-Test split
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_credit, y_credit, stratify=y_credit, test_size=0.2, random_state=42)

# SMOTE Oversampling
Xc_train_resampled, yc_train_resampled = smote.fit_resample(Xc_train, yc_train)


# Train Models (Logistic Regression + XGBoost)

In [5]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    f1 = f1_score(y_test, y_pred)
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    auc_pr = auc(recall, precision)
    
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-PR: {auc_pr:.4f}")
    return f1, auc_pr


# Train and Evaluate – E-Commerce Dataset

In [6]:
print("=== E-Commerce Dataset ===")

# Logistic Regression
logit_fraud = LogisticRegression(max_iter=1000, random_state=42)
logit_fraud.fit(Xf_train_resampled, yf_train_resampled)
print("\nLogistic Regression:")
evaluate_model(logit_fraud, Xf_test, yf_test)

# XGBoost
xgb_fraud = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_fraud.fit(Xf_train_resampled, yf_train_resampled)
print("\nXGBoost:")
evaluate_model(xgb_fraud, Xf_test, yf_test)


=== E-Commerce Dataset ===

Logistic Regression:
              precision    recall  f1-score   support

           0       0.91      0.57      0.70     27393
           1       0.10      0.46      0.17      2830

    accuracy                           0.56     30223
   macro avg       0.51      0.52      0.43     30223
weighted avg       0.84      0.56      0.65     30223

Confusion Matrix:
 [[15667 11726]
 [ 1516  1314]]
F1 Score: 0.1656
AUC-PR: 0.0990


c:\Users\addis\miniconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [22:48:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost:
              precision    recall  f1-score   support

           0       0.95      0.72      0.82     27393
           1       0.18      0.60      0.28      2830

    accuracy                           0.71     30223
   macro avg       0.56      0.66      0.55     30223
weighted avg       0.87      0.71      0.77     30223

Confusion Matrix:
 [[19737  7656]
 [ 1124  1706]]
F1 Score: 0.2799
AUC-PR: 0.2900


(0.2798556430446194, np.float64(0.28997336457963985))

#  Train and Evaluate – Credit Card Dataset

In [7]:
print("=== Credit Card Dataset ===")

# Logistic Regression
logit_credit = LogisticRegression(max_iter=1000, random_state=42)
logit_credit.fit(Xc_train_resampled, yc_train_resampled)
print("\nLogistic Regression:")
evaluate_model(logit_credit, Xc_test, yc_test)

# XGBoost
xgb_credit = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_credit.fit(Xc_train_resampled, yc_train_resampled)
print("\nXGBoost:")
evaluate_model(xgb_credit, Xc_test, yc_test)


=== Credit Card Dataset ===


c:\Users\addis\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     56864
           1       0.12      0.91      0.22        98

    accuracy                           0.99     56962
   macro avg       0.56      0.95      0.61     56962
weighted avg       1.00      0.99      0.99     56962

Confusion Matrix:
 [[56226   638]
 [    9    89]]
F1 Score: 0.2158
AUC-PR: 0.7892


c:\Users\addis\miniconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [22:49:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.77      0.87      0.82        98

    accuracy                           1.00     56962
   macro avg       0.89      0.93      0.91     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix:
 [[56839    25]
 [   13    85]]
F1 Score: 0.8173
AUC-PR: 0.8701


(0.8173076923076923, np.float64(0.8701005607084085))